In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf ,plot_pacf

from sklearn.metrics import( mean_squared_error,mean_absolute_error,mean_absolute_percentage_error,r2_score)


In [2]:
df=pd.read_csv(r"C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\time_series.csv",parse_dates=["Date"])
df=df.set_index("Date")
df=df.asfreq("MS")
df.head()

,KCL
Date,
1967-01-01,25.5
1967-02-01,25.5
1967-03-01,25.5
1967-04-01,25.5
1967-05-01,25.5


In [3]:
df.isna().sum()

KCL    1
dtype: int64

In [4]:
df=df.dropna()

In [5]:
y=df["KCL"]

In [6]:
y.isna().sum()

np.int64(0)

In [7]:
y=y.dropna()

In [8]:
train_size=int(len(y)*0.8)
y_train=y.iloc[:train_size]
y_test=y.iloc[train_size:]


In [9]:
df.isna().sum()

KCL    0
dtype: int64

In [10]:
df=df.dropna()

In [11]:
#grid search
warnings.filterwarnings("ignore")

#meilleur ordre trouvé avec arima
order = (2, 1, 3)

#parametres saisonniers
P=range(0,2)
D=range(0,2)
Q=range(0,2)
s=12#donnees mensuelles

results=[]

count = 1
total = len(P) * len(D) * len(Q)


for seasonal in itertools.product(P,D,Q):
    seasonal_order=(
        seasonal[0],
        seasonal[1],
        seasonal[2],
        s
    )

    print(f"Modèle {count}/{total} : order={order}, seasonal={seasonal_order}")
    try:
        model=SARIMAX(
            y_train,
            # exog=X_train,#sans vars exogenes
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False

        )
        fitted=model.fit(disp=False)

        results.append(
            {
                "Order":order,
                "Seasonal_Order":seasonal_order,
                "AIC":fitted.aic,
                "BIC":fitted.bic
            }
        )
    except Exception as e:
        print(e)

    count += 1
results=pd.DataFrame(results)
results=results.sort_values(by="AIC")
print(results.head(10))

Modèle 1/8 : order=(2, 1, 3), seasonal=(0, 0, 0, 12)
Modèle 2/8 : order=(2, 1, 3), seasonal=(0, 0, 1, 12)
Modèle 3/8 : order=(2, 1, 3), seasonal=(0, 1, 0, 12)
Modèle 4/8 : order=(2, 1, 3), seasonal=(0, 1, 1, 12)
Modèle 5/8 : order=(2, 1, 3), seasonal=(1, 0, 0, 12)
Modèle 6/8 : order=(2, 1, 3), seasonal=(1, 0, 1, 12)
Modèle 7/8 : order=(2, 1, 3), seasonal=(1, 1, 0, 12)
Modèle 8/8 : order=(2, 1, 3), seasonal=(1, 1, 1, 12)
       Order Seasonal_Order          AIC          BIC
3  (2, 1, 3)  (0, 1, 1, 12)  4704.073084  4734.062023
7  (2, 1, 3)  (1, 1, 1, 12)  4709.315197  4743.588270
1  (2, 1, 3)  (0, 0, 1, 12)  4754.687293  4784.831220
5  (2, 1, 3)  (1, 0, 1, 12)  4756.361939  4790.812141
4  (2, 1, 3)  (1, 0, 0, 12)  4771.052934  4801.222362
0  (2, 1, 3)  (0, 0, 0, 12)  4850.073379  4876.041000
6  (2, 1, 3)  (1, 1, 0, 12)  4905.012785  4935.027795
2  (2, 1, 3)  (0, 1, 0, 12)  5223.504866  5249.342518


In [12]:
#meilleur modele
best_order=tuple(results.iloc[0]["Order"])
best_seasonal=tuple(results.iloc[0]["Seasonal_Order"])
print("Best order :", best_order)
print("Best seasonal order :", best_seasonal)

Best order : (2, 1, 3)
Best seasonal order : (0, 1, 1, 12)


In [13]:
model = SARIMAX(
    y_train,
    # exog=X_train,
    order=(2, 1, 3),
    seasonal_order=(0,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

In [14]:
model_fit=model.fit(disp=False)
print(model_fit.summary())

                                      SARIMAX Results                                       
Dep. Variable:                                  KCL   No. Observations:                  565
Model:             SARIMAX(2, 1, 3)x(0, 1, [1], 12)   Log Likelihood               -2345.037
Date:                              Thu, 06 Aug 2026   AIC                           4704.073
Time:                                      12:35:08   BIC                           4734.062
Sample:                                  01-01-1967   HQIC                          4715.805
                                       - 01-01-2014                                         
Covariance Type:                                opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.5840      0.018     32.033      0.000       0.548       0.620
ar.L2         -0.92

In [15]:
#previsions
forecast=model_fit.forecast(
    steps=len(y_test),
    # exog=X_test
)

In [16]:
print("NaN dans y_test :", y_test.isna().sum())

# Si forecast est une Series pandas
print("NaN dans forecast :", forecast.isna().sum())

print(forecast)

NaN dans y_test : 0
NaN dans forecast : 0
2014-02-01    390.426291
2014-03-01    392.399103
2014-04-01    401.278918
2014-05-01    408.462915
2014-06-01    403.445840
                 ...    
2025-07-01    486.180029
2025-08-01    489.651770
2025-09-01    487.872123
2025-10-01    488.138969
2025-11-01    488.577850
Freq: MS, Name: predicted_mean, Length: 142, dtype: float64


In [17]:
print("len(y_train) :", len(y_train))
print("len(y_test) :", len(y_test))
print("len(forecast) :", len(forecast))

print("NaN y_test :", y_test.isna().sum())
print("NaN forecast :", forecast.isna().sum())

print("Index y_test :", y_test.index[0], "->", y_test.index[-1])
print("Index forecast :", forecast.index[0], "->", forecast.index[-1])

len(y_train) : 565
len(y_test) : 142
len(forecast) : 142
NaN y_test : 0
NaN forecast : 0
Index y_test : 2014-02-01 00:00:00 -> 2025-12-01 00:00:00
Index forecast : 2014-02-01 00:00:00 -> 2025-11-01 00:00:00


In [18]:
y_test=y_test.dropna()
forecast=forecast.dropna()

In [19]:
# Aligner les deux séries
y_test_aligned, forecast_aligned = y_test.align(forecast, join="inner")

print(len(y_test_aligned), len(forecast_aligned))

mae = mean_absolute_error(y_test_aligned, forecast_aligned)
mse = mean_squared_error(y_test_aligned, forecast_aligned)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_test_aligned, forecast_aligned)
r2 = r2_score(y_test_aligned, forecast_aligned)

print("MAE :", mae)
print("RMSE :", rmse)
print(f"MAPE : {mape*100:.2f}%")
print("R² :", r2)

141 141
MAE : 185.74515768647467
RMSE : 214.66684209401902
MAPE : 59.54%
R² : -0.13221413681112892
